In [3]:
import pandas as pd
cages = pd.read_csv("cages_ANONYMIZED.csv", dtype=str)
maintenance = pd.read_csv("/content/Cage Maintenance_ANONYMIZED.csv", dtype=str)
vendors = pd.read_csv("/content/Sales Patriot Project Vendor File_ANONYMIZED.csv", dtype=str)

"""
2 flaws:
  1) inconsistent casing
  2) inconsistent spacing
"""
cages["id"]              = cages["id"].str.strip().str.upper()
maintenance["CageCode"]   = maintenance["CageCode"].str.strip().str.upper()
vendors["VendorId"]       = vendors["VendorId"].str.strip()
"""
2 flaws:
  1) vendorCode has trailing .0, we strip this
  2) vendorId has 6 digits, to match this, fill vendorCode to ensure it has 6 digits and matches VendorId
"""
maintenance["VendorCode"] = (maintenance["VendorCode"].str.replace(r"\.0$", "", regex=True).str.zfill(6))

"""
  Create Linked Vendor-Manufacturer List
"""
# VendorId whose name is null = placeholder vendor
nameless_vendor_ids = set(vendors[ vendors["VendorName"].isna() ]["VendorId"].dropna())
#for rows where mask is True, set VendorCode col to null
mask = maintenance["VendorCode"].isin(nameless_vendor_ids)
maintenance.loc[mask, "VendorCode"] = pd.NA

#maintenance info with cage info attached
mm = pd.merge(maintenance, cages, left_on="CageCode", right_on="id", how="left")

#isna() will return a list of true false, true if null, only extract those that are null for orphans
orphan_manufacturers = mm[ mm["VendorCode"].isna() ].copy()
#notna() returns true if not null, only extract those that are not null for linked.
linked_firms = mm[ mm["VendorCode"].notna() ].copy()
#add link id column to only vendor-manufacturer links
linked_firms["linkID"] = linked_firms["VendorCode"]

#now we wan tto make sure we get parents that are unique
parent_VendorId = linked_firms["VendorCode"].unique()
parents = vendors[ vendors["VendorId"].isin(parent_VendorId) ].copy()
parents["linkID"] = parents["VendorId"]

parents = parents.drop_duplicates(subset="VendorId")

#upon inspection, parents/vendors have the following information
parents_clean = parents[["VendorName", "VendorId", "EmailAddress", "Phone", "Contact", "linkID"]].copy()
parents_clean.columns = ["company_name", "cage", "email", "phone", "contact_name", "linkID"]

#upon inspection, children/manufacturers do not have email, phone or contact_name column, set to null
children_clean = linked_firms[["company", "id", "linkID"]].copy()
children_clean.columns = ["company_name", "cage", "linkID"]
children_clean["email"] = pd.NA
children_clean["phone"] = pd.NA
children_clean["contact_name"] = pd.NA

#stack the parent and child rows now
linked_list = pd.concat([parents_clean, children_clean], ignore_index=True)
final_cols = ["company_name", "cage", "email", "phone", "contact_name", "linkID"]
linked_list = linked_list[final_cols]


"""
  Create Orphan Vendors and Orphan Manufacturers List
"""

#isin returns true if the vendor is a parent, we kkeep the negation which implies vendor is an orphan
orphan_vendors = vendors[ ~vendors["VendorId"].isin(parent_VendorId) ].copy()


linked_cages = set(linked_list["cage"].dropna())

#edge case: some manufacturers will be listed as null somewhere, but somewhere will be linked to a vender.
orphan_manufacturers = orphan_manufacturers[
    ~( orphan_manufacturers["id"].notna() & orphan_manufacturers["id"].isin(linked_cages) )
].copy()


#upon inspection, orphan parents/vendors have the following information
orphan_vendors_clean = orphan_vendors[["VendorName", "VendorId", "EmailAddress", "Phone", "Contact"]].copy()
orphan_vendors_clean.columns = ["company_name", "cage", "email", "phone", "contact_name"]

#upon inspection, orphan manufacturers/children have the following information
orphan_manufacturers_clean = orphan_manufacturers[["company", "id"]].copy()
orphan_manufacturers_clean.columns = ["company_name", "cage"]
orphan_manufacturers_clean["email"] = pd.NA
orphan_manufacturers_clean["phone"] = pd.NA
orphan_manufacturers_clean["contact_name"] = pd.NA


# manufacturers: every maintenance manufacturer is either linked or orphan, exactly once
print("linked manufacturers (children):", len(children_clean))       # 8492
print("orphan manufacturers (deduped):", len(orphan_manufacturers_clean))  # 11889
print("sum:", len(children_clean) + len(orphan_manufacturers_clean))
print("original mm rows:", len(mm))                                   # 20386
print("difference (should = 5, the dropped dupes):", len(mm) - (len(children_clean) + len(orphan_manufacturers_clean)))

#vendors
print("parent vendors (in linked):", len(parents_clean))    # 5091
print("orphan vendors:", len(orphan_vendors_clean))          # 13227
print("sum:", len(parents_clean) + len(orphan_vendors_clean))
print("total vendors:", len(vendors))                        # 18318

linked_list.to_csv("linked_vendor_manufacturers.csv", index=False)
orphan_manufacturers_clean.to_csv("orphan_manufacturers.csv", index=False)
orphan_vendors_clean.to_csv("orphan_vendors.csv", index=False)


linked manufacturers (children): 8492
orphan manufacturers (deduped): 11889
sum: 20381
original mm rows: 20386
difference (should = 5, the dropped dupes): 5
parent vendors (in linked): 5091
orphan vendors: 13227
sum: 18318
total vendors: 18318
